# Convert Gaia XP mean spectra to sampled spectra

GaiaXPy provides the function `convert` to evaluate continuous XP mean spectra on a pseudo-wavelength grid.

**Contents**

- [Input spectra](#gaiaxpy-input-spectra)
- [Default sampling](#gaiaxpy-default-sampling)
- [Custom sampling](#gaiaxpy-custom-sampling)
- [BP and RP plots](#gaiaxpy-bp-and-rp-plots)
- [Additional options](#gaiaxpy-additional-options)
- [Compare with and without truncation](#gaiaxpy-truncation-comparison)
- [Output files](#gaiaxpy-output-files)

<a id="gaiaxpy-input-spectra" name="gaiaxpy-input-spectra"></a>

## Input spectra

The input consists of continuous XP mean spectra with BP/RP coefficients, their errors and correlations. Local inputs can be CSV, ECSV, FITS, XML/VOTable, including supported gzip files and Archive ZIP downloads as obtained from the Gaia Archive. GaiaXPy also accepts a pandas DataFrame or a pandas Series. For a source-ID list, tuple, or ADQL query, GaiaXPy retrieves the spectra directly from the Archive.

**In order to run this tutorial, replace `spectra_input` with an existing file.**

In [ ]:
from gaiaxpy import convert, plot_spectra

spectra_input = "path/to/dr3_xp_mean_spectra.ecsv"

<a id="gaiaxpy-default-sampling" name="gaiaxpy-default-sampling"></a>

## Default sampling

The function returns a DataFrame and a pseudo-wavelength array. The `xp` column identifies BP or RP. The default grid has 600 points from 0 to 60. Both endpoints are included. This default grid is used when `sampling` is omitted. If a passband is unavailable, its flux and flux-error cells are `None`.

In [ ]:
spectra, pseudo_wavelength = convert(spectra_input, save_file=False)
spectra[["source_id", "xp", "flux", "flux_error"]].head()

# Inspect the first five sampling positions.
pseudo_wavelength[:5]

print(f"{pseudo_wavelength.size} sampling positions")

<a id="gaiaxpy-custom-sampling" name="gaiaxpy-custom-sampling"></a>

## Custom sampling

Custom sampling requires a non-empty, ascending NumPy array of pseudo-wavelengths within [−10, 70]. The example retains the default endpoints but requests 301 points. The returned pseudo-wavelength array gives the sampling positions for the matching flux rows.

In [ ]:
import numpy as np

grid = np.linspace(0.0, 60.0, 301)
custom_spectra, custom_sampling = convert(spectra_input, sampling=grid, save_file=False)

print(custom_sampling[[0, -1]], custom_sampling.size)

<a id="gaiaxpy-bp-and-rp-plots" name="gaiaxpy-bp-and-rp-plots"></a>

## BP and RP plots

The example passes one source's rows and the sampling array to `plot_spectra`. With `multi=True`, BP and RP appear in the same plot, but in separate panels.

In [ ]:
source_id = spectra["source_id"].iloc[0]
plot_spectra(spectra.loc[spectra["source_id"] == source_id], pseudo_wavelength, multi=True)

<a id="gaiaxpy-additional-options" name="gaiaxpy-additional-options"></a>

## Additional options

| Option | Default | Effect |
| --- | --- | --- |
| `truncation` | `False` | `True` selects the recommended number of basis functions from `bp_n_relevant_bases` and `rp_n_relevant_bases`. |
| `with_correlation` | `False` | `True` includes a column with the packed correlations between output samples. |
| `disable_info` | `False` | `True` suppresses routine messages and progress bars. |

<a id="gaiaxpy-truncation-comparison" name="gaiaxpy-truncation-comparison"></a>

## Compare with and without truncation

The default result above uses all coefficients. This example repeats the calculation using `truncation=True` and displays both results for the same source on the default sampling grid. Truncation requires the relevant-basis counts in the input. Each available photometer has its own panel.

In [ ]:
import matplotlib.pyplot as plt

truncated_spectra, truncated_sampling = convert(spectra_input, truncation=True, save_file=False)

source_id = spectra["source_id"].iloc[0]
full_rows = spectra.loc[spectra["source_id"] == source_id]
truncated_rows = truncated_spectra.loc[truncated_spectra["source_id"] == source_id]

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True)
for ax, band in zip(axes, ("BP", "RP")):
    ax.set(title=band, xlabel="Pseudo-wavelength", ylabel="Flux (electrons/s)")
    full_flux = full_rows.loc[full_rows["xp"] == band, "flux"].iloc[0]
    truncated_flux = truncated_rows.loc[truncated_rows["xp"] == band, "flux"].iloc[0]
    if full_flux is None or truncated_flux is None:
        ax.text(0.5, 0.5, "Spectrum unavailable", transform=ax.transAxes,
                ha="center", va="center")
        continue
    ax.plot(pseudo_wavelength, full_flux, label="All coefficients")
    ax.plot(truncated_sampling, truncated_flux, "--", label="Truncated")
    ax.legend()

fig.suptitle(f"Source {source_id}")
fig.tight_layout()
plt.show()

<a id="gaiaxpy-output-files" name="gaiaxpy-output-files"></a>

## Output files

The example sets the output filename stem and format.

```python
convert(spectra_input, save_file=True, output_file="sampled_xp_spectra", output_format="ecsv")
```

The default filename stem is `output_spectra`. `output_path` sets the destination directory. `output_format` accepts CSV, ECSV, FITS or XML/VOTable. If `output_format` is omitted, GaiaXPy uses the underlying input format, or CSV for pandas and Archive selections. CSV sampled-spectrum output writes separate `_sampling.csv` and `_metadata.csv` companions. The rest of the formats include the results along with the metadata and sampling in a single output file.